# Dataset Budget

## Import Semua Packages/Library yang Digunakan

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re

## Data Wrangling

### Gathering Data

In [2]:
# load dataset budget
file_id = "11xx2Wvlx0KUbBi2bBqVo6gYZiaLd7f6g"
url = f"https://drive.google.com/uc?export=download&id={file_id}"

df = pd.read_csv(url)

# melihat data awal
df.head()

,budget_id,user_id,income_id,needs_amount,wants_amount,investment_amount,income_amount,budget_limit,source,income_date
0,BGT01352,USR053,INC06728,3400000,2250000,1600000,7250000,5650000,salary,2023-01-10 00:38
1,BGT06784,USR016,INC00797,650000,400000,300000,1400000,1050000,scholarship,2024-05-25 12:49
2,BGT05527,USR050,INC01634,2150000,1150000,550000,3800000,3300000,freelance,2023-04-25 15:24
3,BGT02948,USR013,INC08009,600000,350000,250000,1200000,950000,scholarship,2023-07-21 06:16
4,BGT03140,USR038,INC06623,2550000,1450000,950000,4950000,4000000,freelance,2024-04-07 04:12


### Asessing Data

In [3]:
# cek dimensi dataset
df.shape

(10040, 10)

In [4]:
# cek tipe data dan jumlah non-null tiap kolom
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10040 entries, 0 to 10039
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   budget_id          10040 non-null  object
 1   user_id            10040 non-null  object
 2   income_id          10040 non-null  object
 3   needs_amount       9830 non-null   object
 4   wants_amount       9826 non-null   object
 5   investment_amount  9706 non-null   object
 6   income_amount      9778 non-null   object
 7   budget_limit       10040 non-null  int64 
 8   source             9934 non-null   object
 9   income_date        9759 non-null   object
dtypes: int64(1), object(9)
memory usage: 784.5+ KB


In [5]:
# cek missing values (NaN) per kolom
df.isnull().sum()

,0
budget_id,0
user_id,0
income_id,0
needs_amount,210
wants_amount,214
investment_amount,334
income_amount,262
budget_limit,0
source,106
income_date,281


In [6]:
# cek duplikasi baris penuh
print("Jumlah baris duplikat:", df.duplicated().sum())

Jumlah baris duplikat: 40


In [7]:
# cek inkonsistensi nilai kategorikal
for col in ['source']:
    print(f"\n{col}:", df[col].unique())


source: ['salary' 'scholarship' 'freelance' 'FREELANCE' 'business' 'bonus'
 'allowance' 'part_time' 'investment_return' 'internship' nan 'PART_TIME'
 'SALARY' 'ALLOWANCE' 'INVESTMENT_RETURN' 'BONUS' 'SCHOLARSHIP' 'BUSINESS'
 'INTERNSHIP']


In [8]:
# frekuensi source
print(df['source'].value_counts())

source
freelance            1698
allowance            1635
part_time            1462
business             1036
salary               1029
bonus                 987
scholarship           828
internship            648
investment_return     412
ALLOWANCE              47
PART_TIME              37
FREELANCE              31
SALARY                 17
BONUS                  17
SCHOLARSHIP            17
BUSINESS               13
INVESTMENT_RETURN      12
INTERNSHIP              8
Name: count, dtype: int64


In [9]:
# cek statistik deskriptif data numerik
df.describe()

,budget_limit
count,1.004000e+04
mean,3.419895e+06
std,1.826738e+06
min,7.000000e+05
25%,1.900000e+06
50%,3.100000e+06
75%,4.750000e+06
max,9.850000e+06


In [10]:
# Deteksi inkonsistensi format tanggal
date_sample = df['income_date'].dropna().astype(str)

fmt_ddmmyyyy   = date_sample.str.match(r'^\d{2}/\d{2}/\d{4}').sum()
fmt_yyyymmdd_s = date_sample.str.match(r'^\d{4}/\d{2}/\d{2}').sum()
fmt_yyyymmdd_d = date_sample.str.match(r'^\d{4}-\d{2}-\d{2}').sum()

print(f'DD/MM/YYYY HH:MM : {fmt_ddmmyyyy} baris')
print(f'YYYY/MM/DD HH:MM:SS: {fmt_yyyymmdd_s} baris')
print(f'YYYY-MM-DD HH:MM:SS (standar) : {fmt_yyyymmdd_d} baris')

DD/MM/YYYY HH:MM : 268 baris
YYYY/MM/DD HH:MM:SS: 158 baris
YYYY-MM-DD HH:MM:SS (standar) : 9333 baris


In [11]:
# Deteksi data numerik tidak valid
numeric_cols = [
    'needs_amount', 'wants_amount', 'investment_amount',
    'income_amount', 'budget_limit'
]

for col in numeric_cols:
    non_numeric = df[pd.to_numeric(df[col], errors='coerce').isna() & df[col].notna()]
    print(f'Kolom {col} - data non numerik : {len(non_numeric)}')

Kolom needs_amount - data non numerik : 136
Kolom wants_amount - data non numerik : 142
Kolom investment_amount - data non numerik : 161
Kolom income_amount - data non numerik : 161
Kolom budget_limit - data non numerik : 0


In [12]:
# cek apakah total budget melebihi income
invalid_budget = df[
    (
        pd.to_numeric(df['needs_amount'], errors='coerce') +
        pd.to_numeric(df['wants_amount'], errors='coerce') +
        pd.to_numeric(df['investment_amount'], errors='coerce')
    ) > pd.to_numeric(df['income_amount'], errors='coerce')
]

print("Jumlah budget melebihi income:", len(invalid_budget))
invalid_budget.head()


Jumlah budget melebihi income: 1107


,budget_id,user_id,income_id,needs_amount,wants_amount,investment_amount,income_amount,budget_limit,source,income_date
2,BGT05527,USR050,INC01634,2150000,1150000,550000,3800000,3300000,freelance,2023-04-25 15:24
63,BGT05707,USR049,INC03730,3900000,2250000,1650000,7750000,6150000,salary,2024-08-31 03:02
65,BGT08101,USR022,INC05222,1700000,1100000,900000,3650000,2800000,NaN,2024-09-06 15:36
67,BGT03018,USR029,INC02343,1900000,1300000,850000,4000000,3200000,part_time,2023-05-26 06:53
79,BGT08435,USR015,INC00156,1400000,950000,700000,3000000,2350000,allowance,2024-02-15 01:37


**Insight:**
- Terdapat missing values pada beberapa kolom numerik dan kategorikal
- Terdapat beberapa baris duplikat pada dataset
- Kolom source memiliki typo dan inkonsistensi penulisan
- Kolom income_date menggunakan format datetime yang tidak konsisten
- Beberapa kolom numerik masih berbentuk string dan perlu dibersihkan
- Ditemukan nilai negatif dan nol pada beberapa nominal budget
- Terdapat inkonsistensi antara total budget dan income_amount
- Ditemukan outlier pada income_amount

### Cleaning Data


In [13]:
# Membuat salinan agar data asli tetap terjaga
df_clean = df.copy()

In [14]:
# Bersihkan & konversi kolom numerik
numeric_cols = [
    'needs_amount', 'wants_amount', 'investment_amount',
    'income_amount', 'budget_limit'
]

def clean_currency(val):
    if pd.isna(val) or str(val).strip() == '':
        return np.nan

    val = str(val).strip().lower()
    val = val.replace('rp', '')
    val = val.replace('.', '')
    val = val.replace(',', '')
    val = val.replace(' ', '')

    try:
        return float(val)
    except:
        return np.nan

for col in numeric_cols:
    df_clean[col] = df_clean[col].apply(clean_currency)

# cek hasil cleaning numerik
print(df_clean[numeric_cols].head())

# cek missing values setelah cleaning
print('\nMissing values setelah cleaning:')
print(df_clean[numeric_cols].isnull().sum())

   needs_amount  wants_amount  investment_amount  income_amount  budget_limit
0     3400000.0     2250000.0          1600000.0      7250000.0     5650000.0
1      650000.0      400000.0           300000.0      1400000.0     1050000.0
2     2150000.0     1150000.0           550000.0      3800000.0     3300000.0
3      600000.0      350000.0           250000.0      1200000.0      950000.0
4     2550000.0     1450000.0           950000.0      4950000.0     4000000.0

Missing values setelah cleaning:
needs_amount         346
wants_amount         356
investment_amount    495
income_amount        423
budget_limit           0
dtype: int64


In [15]:
# Standarisasi kolom income_date menjadi datetime

def parse_date(val):
    if pd.isna(val) or str(val).strip() == '':
        return pd.NaT

    for fmt in (
        '%Y-%m-%d %H:%M',
        '%Y-%m-%d %H:%M:%S',
        '%Y/%m/%d %H:%M:%S',
        '%d/%m/%Y %H:%M'
    ):
        try:
            return pd.to_datetime(val, format=fmt)
        except:
            continue

    return pd.NaT

df_clean['income_date'] = df_clean['income_date'].apply(parse_date)

before = len(df_clean)

df_clean.dropna(subset=['income_date'], inplace=True)

print(f'Baris tanggal invalid dihapus : {before - len(df_clean)}')
print(f'Tipe kolom income_date : {df_clean["income_date"].dtype}')

Baris tanggal invalid dihapus : 439
Tipe kolom income_date : datetime64[ns]


In [16]:
# Standarisasi kolom source
df_clean['source'] = (
    df_clean['source']
    .astype(str)
    .str.strip()
    .str.lower()
)

# ubah string 'nan' menjadi NaN asli
df_clean['source'] = df_clean['source'].replace('nan', np.nan)

# cari modus source
modus_source = df_clean['source'].mode()[0]

# isi missing value dengan modus
df_clean['source'] = df_clean['source'].fillna(modus_source)

print(df_clean['source'].unique())

['salary' 'scholarship' 'freelance' 'bonus' 'business' 'allowance'
 'part_time' 'internship' 'investment_return']


In [17]:
# Hapus data duplikat
before = len(df_clean)

df_clean.drop_duplicates(inplace=True)

print(f'Duplikat dihapus : {before - len(df_clean)}')
print(f'Sisa data : {len(df_clean)}')

Duplikat dihapus : 37
Sisa data : 9564


In [18]:
# Validasi budget tidak melebihi income
df_clean = df_clean[
    (
        df_clean['needs_amount'] +
        df_clean['wants_amount'] +
        df_clean['investment_amount']
    ) <= df_clean['income_amount']
].copy()

print("Dataset setelah validasi budget:", df_clean.shape)

Dataset setelah validasi budget: (7073, 10)


In [19]:
# hitung ulang budget_limit
df_clean['budget_limit'] = (
    df_clean['needs_amount'] +
    df_clean['wants_amount']
)

In [20]:
print('RINGKASAN SETELAH CLEANING')
print(f'Shape              : {df_clean.shape}')
print(f'Missing values     : {df_clean.isnull().sum().sum()}')
print(f'Duplikat           : {df_clean.duplicated().sum()}')
print(f'Tipe income_date   : {df_clean["income_date"].dtype}')

RINGKASAN SETELAH CLEANING
Shape              : (7073, 10)
Missing values     : 0
Duplikat           : 0
Tipe income_date   : datetime64[ns]


In [21]:
# preview dataset bersih

df_clean.head(10)

,budget_id,user_id,income_id,needs_amount,wants_amount,investment_amount,income_amount,budget_limit,source,income_date
0,BGT01352,USR053,INC06728,3400000.0,2250000.0,1600000.0,7250000.0,5650000.0,salary,2023-01-10 00:38:00
1,BGT06784,USR016,INC00797,650000.0,400000.0,300000.0,1400000.0,1050000.0,scholarship,2024-05-25 12:49:00
3,BGT02948,USR013,INC08009,600000.0,350000.0,250000.0,1200000.0,950000.0,scholarship,2023-07-21 06:16:00
4,BGT03140,USR038,INC06623,2550000.0,1450000.0,950000.0,4950000.0,4000000.0,freelance,2024-04-07 04:12:00
6,BGT01706,USR026,INC01374,1900000.0,1150000.0,650000.0,3700000.0,3050000.0,freelance,2024-11-20 09:55:00
7,BGT01104,USR068,INC06927,4350000.0,2700000.0,1650000.0,8700000.0,7050000.0,freelance,2024-04-30 16:24:00
8,BGT07466,USR040,INC00896,1500000.0,1100000.0,650000.0,3300000.0,2600000.0,freelance,2024-11-29 03:02:00
10,BGT02716,USR064,INC07291,4700000.0,2550000.0,1300000.0,8550000.0,7250000.0,bonus,2023-03-31 09:31:00
11,BGT09094,USR013,INC09837,650000.0,500000.0,300000.0,1450000.0,1150000.0,scholarship,2023-06-23 12:06:00
12,BGT07765,USR059,INC03771,1950000.0,1350000.0,1150000.0,4450000.0,3300000.0,business,2024-09-13 00:58:00


In [22]:
# simpan dataset bersih
df_clean.to_csv("budget_management_clean.csv", index=False)